# One-Port S11 Sweep

Measures S11 of the MM4250 at a list of RF channels and saves each as a QCoDeS run in `mm4250_sweeps.db`. The same as the two-port notebook, for reflection only.

**Files it needs**, all in the same folder as this notebook: `vna_measure.py`, `sweep_db.py`, `read_db.py` and `plots.py`. The drivers are found by walking up to the folder that holds `drivers/`.

**What it writes**, beside this notebook:

```
mm4250_sweeps.db                                                        every sweep, always
Sweeps/<serials>/<date>/<temp>/<setup>_<cal|uncal>/RF<n>_run<id>.s1p   only with touchstone=True
figures/<serials>/<date>/<temp>/<name>.png                              only when a plot is saved
```

Each sweep returns its run ids, which `plot_sweep`, `load_run` and `export_touchstone` take. Whether a sweep was calibrated is read off the VNA and recorded -- runs are grouped as `cal` or `uncal` automatically.

**Instruments:** this notebook connects its own `ksvna`/`switch`. To sweep from `QCodesMeasurmentFramework.ipynb`'s kernel instead, skip the connect cell and call `run_oneport_sweep(...)` there. Never both in one kernel: the switch's USB connection is exclusive.

## Imports

Puts this notebook's folder on the path (for `vna_measure`, `sweep_db`, `read_db`, `plots`), walks up to find `drivers/`, and imports the VNA and switch drivers plus everything the later cells use. Prints both paths so you can check them.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

# The drivers live in whichever repo root is above this notebook -- this
# folder if it's the mm4250-switch repo, or the framework repo if
# these files were copied into users/<name>/. Walk up until we find it.
try:
    here = Path(_dh[0])  # Jupyter sets _dh[0] to this notebook's directory
except NameError:
    here = Path.cwd()
if str(here) not in sys.path:
    sys.path.insert(0, str(here))       # vna_measure / sweep_db live here

root = here
while not (root / "drivers").is_dir() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))       # drivers/ live here
print(f"notebook folder: {here}")
print(f"drivers from:    {root}")

from drivers.KeysightVNA_driver import KeysightP5004B
from drivers.MM4250_QCodes_driver import MM4250
from vna_measure import setup_sweep, sweep_settings, measure_s11
from sweep_db import run_oneport_sweep
from plots import plot_measurement, plot_sweep, summarize
from read_db import list_runs, load_run, export_touchstone

## Connect the VNA and switch

Creates `ksvna` (Keysight P5004B, over the network) and `switch` (MM4250, over USB). Connecting the switch **forces `ALL_OPEN`**.

In [ ]:
ksvna = KeysightP5004B("ksvna", "TCPIP0::QTSF-Measurement::hislip_PXI0_CHASSIS1_SLOT1_INDEX0::INSTR")
switch = MM4250("switch")

## Sweep setup

Sets frequency range, points, IF bandwidth and power -- only what you pass changes -- and always sets trigger source `IMM` and RF output on, which a sweep can't finish without. Power is capped at 0 dBm. Changing range, points or IF bandwidth invalidates any VNA calibration.

In [ ]:
# Sweep setup -- only the settings you pass are changed, plus the
# trigger source (IMM) and RF output, which a sweep can't finish without.
setup_sweep(start=1e9, stop=10e9, points=1001, if_bandwidth=1e3, power=-20, vna=ksvna)

## A single measurement

Measures S11 at one channel and **saves nothing**. `measure_s11(3)` sets the switch to RF3 and returns `freq` (Hz) and `s11` (a complex array). `summarize` prints min, max and marker values; `plot_measurement` plots it in dB and takes `title=`, `phase=True`, `xlim=`, `ylim=`, `colors=`, `labels=` and `save="name.png"` (saved into `figures/`).

In [ ]:
# A single measurement, nothing saved:
freq, s11 = measure_s11(3, vna=ksvna, switch=switch)

summarize(freq, s11, title="RF3")
ax = plot_measurement(freq, s11, title="RF3")


## Batch sweep

Measures S11 at every channel in `channels` and saves each as a run. Edit `channels`, `date_str`, `temp_str` and `switch_serials` each time; add `setup="RF3"` to name the cabling (groups become `RF3_cal` / `RF3_uncal`).

Returns `runs`, the run ids. Add `touchstone=True` to also write `.s1p` files, or write them later with `export_touchstone(runs)`.

In [ ]:
# Edit these for each run, then save the whole set:
channels = [1, 3, 5]         # any subset of 1-6; use list(range(1, 7)) for all of them
date_str = "YYYYMMDD"        # e.g. "20260917"
temp_str = "295K"            # e.g. "295K", "3K", "25mK"
switch_serials = "SN0001"    # the switch under test, e.g. "0030"

runs = run_oneport_sweep(channels, date_str, temp_str, switch_serials, vna=ksvna, switch=switch)   # touchstone=True for .s1p files
print(f"Sweep complete: runs {runs}")

## Look at the whole sweep

Every channel overlaid on one axes, read back from the database -- so any runs from any session work too (`plot_sweep([12, 13, 14])`), as does a folder of `.s1p` files. Options:

```python
plot_sweep(runs, xlim=(0, 6), ylim=(-40, 0), colors=["black", "tab:red"],
           labels=["RF1", "RF3"], title="S11, SN0077", save=True)   # -> figures/<serials>/<date>/<temp>/
```

`save=True` names the file for you; `save="name.pdf"` picks the name and format. One run on its own: `plot_measurement(run=12)`.

In [ ]:
# Every channel overlaid on one axes -- the comparison view.
# Read back from the database, so it also works on runs from any
# earlier session -- plot_sweep([12, 13, 14]) -- or a folder of files.
ax = plot_sweep(runs)


## Close

Backs up the database to `db_backups/` (nothing is written if no runs are new since the last backup), then turns the VNA's RF output off, opens every switch channel, and releases both instruments.

In [ ]:
from read_db import backup_db
backup_db()          # snapshot to db_backups/ beside the .db; skipped if nothing new

ksvna.output(False)
switch.open_all()
ksvna.close()
switch.close()